# Knee Rehab App: Colab version (Project 2, 24-679)

**How to run:** `Runtime → Run all`. The app appears under step 5 (or click **Open the app in a new tab**).
Don't open `localhost` or `127.0.0.1` in your browser: the server runs on Google's machine, not your laptop. If step 5's link doesn't work, run step 6 for a public link.

| Step | What it does |
|---|---|
| 1 | Settings: where the database lives, optional OpenAI key |
| 2–4 | The code files (`rehab_node.py`, `test_server.py`, `test_page.html`) |
| 5 | Starts the app and gives you a link (only you can open it) |
| 6 | *Optional:* a public link for the Figma frontend or teammates |
| 7 | Example: send a detection result from Python (for the CV part) |
| 8 | Simulated patients: fills the database and makes the plan-model dataset |

**Editing code:** change the file in cell 2, 3 or 4 → run that cell → run cell 5 again (it restarts the server).
Save your changes with `File → Save a copy in Drive`, or tell the team which cell you changed.

**Data:** by default the database is deleted when the Colab session ends. Tick `KEEP_DATA_ON_DRIVE` in step 1 to keep it in your own Google Drive.

In [ ]:
#@title 1. Settings
KEEP_DATA_ON_DRIVE = False  #@param {type:"boolean"}
import os
os.makedirs("/content/rehab_node", exist_ok=True)
if KEEP_DATA_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/rehab_data", exist_ok=True)
    os.environ["REHAB_DB"] = "/content/drive/MyDrive/rehab_data/rehab_test.db"
else:
    os.environ["REHAB_DB"] = "/content/rehab_node/rehab_test.db"
print("Database:", os.environ["REHAB_DB"])

# Optional: add a Colab secret named OPENAI_API_KEY (key icon on the left) for LLM plan drafts.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ.setdefault("LLM_MODEL", "gpt-6.2-sol")
    print("OpenAI key loaded")
except Exception:
    print("No OpenAI key: plan drafts use the rule-based version")

## 2. `rehab_node.py`: logic + database (daily pain rule, weekly summary, plan draft, PT approval)

In [ ]:
%%writefile /content/rehab_node/rehab_node.py
"""Daily exercise node: pain check-in, rep adjustment, session logging, weekly summary.

Flow:
  PT plan (set_plan) -> daily_checkin(pain) -> today's adjusted plan
  -> movement analysis sends results -> log_exercise(...)
  -> end of week: weekly_summary(...) -> JSON for the plan-adjustment step
"""
import datetime as dt
import json
import sqlite3
from statistics import mean

DB_PATH = "rehab.db"

# Daily pain rule: (highest pain in band, share of planned reps, action).
# The PT could later change these per patient.
PAIN_BANDS = [
    (3, 1.0, "as_planned"),
    (5, 0.7, "reduced"),
    (6, 0.5, "reduced"),
]
STOP_PAIN = 7  # pain at or above this: no exercise today, urgent alert to PT

SCHEMA = """
CREATE TABLE IF NOT EXISTS patients (
    patient_id    TEXT PRIMARY KEY,
    name          TEXT,
    surgery_date  TEXT,
    operated_side TEXT,
    goal          TEXT
);
CREATE TABLE IF NOT EXISTS plan_items (
    plan_item_id  INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id    TEXT NOT NULL,
    exercise_id   TEXT NOT NULL,
    exercise_name TEXT NOT NULL,
    sets          INTEGER NOT NULL,
    reps          INTEGER NOT NULL,
    start_date    TEXT NOT NULL,
    end_date      TEXT,              -- NULL = current plan
    approved_by   TEXT
);
CREATE TABLE IF NOT EXISTS checkins (
    checkin_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id    TEXT NOT NULL,
    date          TEXT NOT NULL,
    pain_today    INTEGER NOT NULL,  -- 0-10, asked before the session
    pain_location TEXT,              -- front / back / inner / outer / whole / none
    action        TEXT NOT NULL,     -- as_planned / reduced / rest
    rep_factor    REAL NOT NULL,
    UNIQUE (patient_id, date)
);
CREATE TABLE IF NOT EXISTS exercise_logs (
    log_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    checkin_id    INTEGER NOT NULL,
    patient_id    TEXT NOT NULL,
    date          TEXT NOT NULL,
    exercise_id   TEXT NOT NULL,
    planned_sets  INTEGER NOT NULL,
    planned_reps  INTEGER NOT NULL,  -- per set, from the PT plan
    target_reps   INTEGER NOT NULL,  -- per set, after today's pain rule
    sets_done     INTEGER NOT NULL,
    reps_done     INTEGER NOT NULL,  -- total across sets
    form_score    REAL,              -- 0-1, mean over reps
    errors_json   TEXT,              -- {"trunk_lean": 3, ...}
    pain_after    INTEGER            -- 0-10, asked after the exercise
);
CREATE TABLE IF NOT EXISTS alerts (
    alert_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id    TEXT NOT NULL,
    date          TEXT NOT NULL,
    tier          TEXT NOT NULL,     -- urgent / review
    reason        TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS adjustment_requests (
    request_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id    TEXT NOT NULL,
    week_end      TEXT NOT NULL,
    summary_json  TEXT NOT NULL,     -- weekly_summary output
    draft_json    TEXT NOT NULL,     -- what the app proposed
    draft_source  TEXT NOT NULL,     -- rules / llm
    status        TEXT NOT NULL,     -- pending / approved / rejected
    final_json    TEXT,              -- what the PT approved (may differ from draft)
    pt_edited     INTEGER,           -- 1 if the PT changed the draft
    pt_note       TEXT,
    decided_at    TEXT
);
CREATE TABLE IF NOT EXISTS intakes (
    intake_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id    TEXT NOT NULL,
    created_at    TEXT NOT NULL,
    answers_json  TEXT NOT NULL,
    summary       TEXT NOT NULL      -- for the PT; rule-based now, LLM later
);
"""

PAIN_LOCATIONS = ["none", "front", "back", "inner", "outer", "whole knee"]

# The 10 exercises the PT can prescribe (all camera-checkable). The ID is what detection sends.
EXERCISES = {
    "heel_slide": "Heel slide",
    "straight_leg_raise": "Straight leg raise",
    "sitting_knee_bend": "Sitting knee bend",
    "standing_knee_bend": "Standing knee bend",
    "seated_knee_extension": "Seated knee extension",
    "sit_to_stand": "Sit-to-stand",
    "mini_squat": "Mini squat",
    "step_up": "Step-up",
    "glute_bridge": "Glute bridge",
    "side_lying_hip_abduction": "Side-lying hip abduction",
}

# Patient goals, lightest to hardest. "targets" = the program the goal needs (sets, reps).
# Starting values: confirm with PTs (add to the PT survey).
GOALS = {
    "home":   {"label": "Get through the day at home without pain", "focus": "range of motion",
               "min_week": 0, "targets": {"heel_slide": (3, 10), "straight_leg_raise": (2, 10),
                                          "sit_to_stand": (2, 10)}},
    "walk":   {"label": "Walk for 30 minutes", "focus": "endurance",
               "min_week": 2, "targets": {"heel_slide": (3, 10), "straight_leg_raise": (3, 10),
                                          "sit_to_stand": (3, 12), "glute_bridge": (3, 12)}},
    "stairs": {"label": "Climb stairs", "focus": "leg strength and control",
               "min_week": 3, "targets": {"heel_slide": (3, 10), "sit_to_stand": (3, 12), "glute_bridge": (3, 12),
                                          "step_up": (3, 10), "mini_squat": (3, 10),
                                          "side_lying_hip_abduction": (2, 12)}},
    "jog":    {"label": "Jog", "focus": "strength and power",
               "min_week": 12, "targets": {"sit_to_stand": (3, 15), "glute_bridge": (3, 15), "step_up": (3, 12),
                                           "mini_squat": (3, 12), "side_lying_hip_abduction": (3, 12),
                                           "standing_knee_bend": (3, 12)}},
}
DEFAULT_GOAL = "home"


def connect(path=DB_PATH):
    con = sqlite3.connect(path, check_same_thread=False)  # Gradio calls from worker threads
    con.row_factory = sqlite3.Row
    con.executescript(SCHEMA)
    return con


def _today(date):
    return date or dt.date.today().isoformat()


def rep_rule(pain_today):
    """Return (action, rep_factor) for today's pain score."""
    if not 0 <= pain_today <= 10:
        raise ValueError("pain_today must be 0-10")
    if pain_today >= STOP_PAIN:
        return "rest", 0.0
    for top, factor, action in PAIN_BANDS:
        if pain_today <= top:
            return action, factor
    return "rest", 0.0


def adjusted_reps(reps, factor):
    """Round half up, never below 1 unless resting (10 -> 7 at 0.7, 10 -> 5 at 0.5)."""
    if factor == 0:
        return 0
    return max(1, int(reps * factor + 0.5))


# ---------- PT side ----------

def add_patient(con, patient_id, name, surgery_date, operated_side, goal=""):
    con.execute(
        "INSERT OR REPLACE INTO patients VALUES (?,?,?,?,?)",
        (patient_id, name, surgery_date, operated_side, goal),
    )
    con.commit()


def set_plan(con, patient_id, items, approved_by, start_date=None):
    """Replace the current plan. items: [{"exercise_id","exercise_name","sets","reps"}]."""
    start = _today(start_date)
    con.execute(
        "UPDATE plan_items SET end_date=? WHERE patient_id=? AND end_date IS NULL",
        (start, patient_id),
    )
    for it in items:
        con.execute(
            "INSERT INTO plan_items (patient_id, exercise_id, exercise_name, sets, reps, start_date, approved_by)"
            " VALUES (?,?,?,?,?,?,?)",
            (patient_id, it["exercise_id"], it["exercise_name"], int(it["sets"]), int(it["reps"]), start, approved_by),
        )
    con.commit()


def current_plan(con, patient_id):
    rows = con.execute(
        "SELECT exercise_id, exercise_name, sets, reps FROM plan_items"
        " WHERE patient_id=? AND end_date IS NULL ORDER BY plan_item_id",
        (patient_id,),
    ).fetchall()
    return [dict(r) for r in rows]


# ---------- Patient side: daily ----------

def daily_checkin(con, patient_id, pain_today, date=None, pain_location="none"):
    """Ask pain before the session; return today's plan with adjusted reps.

    pain_location is stored for the PT and the weekly summary; it does not change reps yet.
    """
    date = _today(date)
    if not current_plan(con, patient_id):
        raise ValueError(f"No plan for {patient_id}. The PT must enter a plan first.")
    action, factor = rep_rule(pain_today)
    con.execute(
        "INSERT INTO checkins (patient_id, date, pain_today, pain_location, action, rep_factor) VALUES (?,?,?,?,?,?)"
        " ON CONFLICT(patient_id, date) DO UPDATE SET pain_today=excluded.pain_today,"
        " pain_location=excluded.pain_location, action=excluded.action, rep_factor=excluded.rep_factor",
        (patient_id, date, pain_today, pain_location, action, factor),
    )
    if action == "rest":
        con.execute(
            "INSERT INTO alerts (patient_id, date, tier, reason) VALUES (?,?,?,?)",
            (patient_id, date, "urgent", f"Pain {pain_today}/10 before session; exercise skipped today"),
        )
    con.commit()
    checkin_id = con.execute(
        "SELECT checkin_id FROM checkins WHERE patient_id=? AND date=?", (patient_id, date)
    ).fetchone()[0]

    today = []
    for it in current_plan(con, patient_id):
        today.append({
            "exercise_id": it["exercise_id"],
            "exercise_name": it["exercise_name"],
            "sets": 0 if action == "rest" else it["sets"],
            "planned_reps": it["reps"],
            "target_reps": adjusted_reps(it["reps"], factor),
        })
    messages = {
        "as_planned": "Your usual plan today.",
        "reduced": "Lighter session today because of your pain.",
        "rest": "Please rest today. We've told your PT.",
    }
    return {"checkin_id": checkin_id, "date": date, "pain_today": pain_today,
            "action": action, "message": messages[action], "exercises": today}


def log_exercise(con, checkin_id, result):
    """Store one exercise result from movement analysis.

    result: {"exercise_id", "sets_done", "reps_done", "form_scores": [0-1 per rep],
             "error_counts": {...}, "pain_after": 0-10}
    """
    ck = con.execute("SELECT * FROM checkins WHERE checkin_id=?", (checkin_id,)).fetchone()
    if ck is None:
        raise ValueError("unknown checkin_id")
    plan = {p["exercise_id"]: p for p in current_plan(con, ck["patient_id"])}
    item = plan.get(result["exercise_id"])
    if item is None:
        raise ValueError(f"{result['exercise_id']} is not in the current plan")

    scores = result.get("form_scores") or []
    form = round(mean(scores), 3) if scores else None
    pain_after = result.get("pain_after")
    con.execute(
        "INSERT INTO exercise_logs (checkin_id, patient_id, date, exercise_id, planned_sets, planned_reps,"
        " target_reps, sets_done, reps_done, form_score, errors_json, pain_after)"
        " VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
        (checkin_id, ck["patient_id"], ck["date"], result["exercise_id"], item["sets"], item["reps"],
         adjusted_reps(item["reps"], ck["rep_factor"]), int(result["sets_done"]), int(result["reps_done"]),
         form, json.dumps(result.get("error_counts") or {}), pain_after),
    )
    if pain_after is not None and pain_after >= STOP_PAIN:
        con.execute(
            "INSERT INTO alerts (patient_id, date, tier, reason) VALUES (?,?,?,?)",
            (ck["patient_id"], ck["date"], "urgent",
             f"Pain {pain_after}/10 after {item['exercise_name']}"),
        )
    con.commit()


# ---------- End of week: input for plan adjustment ----------

def weekly_summary(con, patient_id, week_end=None, days=7):
    """Aggregate the last `days` days into one JSON for the plan-adjustment step."""
    end = dt.date.fromisoformat(_today(week_end))
    start = end - dt.timedelta(days=days - 1)
    s, e = start.isoformat(), end.isoformat()

    checkins = [dict(r) for r in con.execute(
        "SELECT date, pain_today, pain_location, action FROM checkins"
        " WHERE patient_id=? AND date BETWEEN ? AND ? ORDER BY date",
        (patient_id, s, e)).fetchall()]
    patient = con.execute("SELECT * FROM patients WHERE patient_id=?", (patient_id,)).fetchone()
    logs = [dict(r) for r in con.execute(
        "SELECT * FROM exercise_logs WHERE patient_id=? AND date BETWEEN ? AND ? ORDER BY date",
        (patient_id, s, e)).fetchall()]
    alerts = [dict(r) for r in con.execute(
        "SELECT date, tier, reason FROM alerts WHERE patient_id=? AND date BETWEEN ? AND ? ORDER BY date",
        (patient_id, s, e)).fetchall()]

    exercises = []
    for item in current_plan(con, patient_id):
        rows = [l for l in logs if l["exercise_id"] == item["exercise_id"]]
        planned_total = item["sets"] * item["reps"]
        forms = [r["form_score"] for r in rows if r["form_score"] is not None]
        pains = [r["pain_after"] for r in rows if r["pain_after"] is not None]
        high_days = len({r["date"] for r in rows if r["pain_after"] is not None and r["pain_after"] >= STOP_PAIN})
        errors = {}
        for r in rows:
            for k, v in json.loads(r["errors_json"] or "{}").items():
                errors[k] = errors.get(k, 0) + v
        half = len(forms) // 2
        exercises.append({
            "exercise_id": item["exercise_id"],
            "exercise_name": item["exercise_name"],
            "planned": f"{item['sets']} x {item['reps']}",
            "days_done": len(rows),
            "reps_done_share": round(mean(min(1.0, r["reps_done"] / max(1, r["planned_sets"] * r["target_reps"])) for r in rows), 2) if rows else 0.0,
            "form_avg": round(mean(forms), 2) if forms else None,
            "form_trend": round(mean(forms[half:]) - mean(forms[:half]), 2) if half >= 1 else None,
            "pain_after_avg": round(mean(pains), 1) if pains else None,
            "pain_after_max": max(pains) if pains else None,
            "high_pain_days": high_days,
            "error_counts": errors,
        })

    locations = {}
    for c in checkins:
        loc = c["pain_location"] or "none"
        locations[loc] = locations.get(loc, 0) + 1

    goal = patient["goal"] if patient and patient["goal"] in GOALS else DEFAULT_GOAL
    week_post_op = None
    if patient and patient["surgery_date"]:
        try:
            week_post_op = max(0, (end - dt.date.fromisoformat(patient["surgery_date"])).days // 7)
        except ValueError:
            pass

    return {
        "patient_id": patient_id,
        "goal": goal,
        "goal_label": GOALS[goal]["label"],
        "week_post_op": week_post_op,
        "week": {"start": s, "end": e},
        "days_checked_in": len(checkins),
        "pain_location_days": locations,
        "days_reduced": sum(c["action"] == "reduced" for c in checkins),
        "days_rest": sum(c["action"] == "rest" for c in checkins),
        "pain_today_avg": round(mean(c["pain_today"] for c in checkins), 1) if checkins else None,
        "pain_today_by_day": {c["date"]: c["pain_today"] for c in checkins},
        "exercises": exercises,
        "alerts": alerts,
    }


# ---------- Plan adjustment draft (sent to the PT, never straight to the patient) ----------

# Starting thresholds; tune them with the PT ground-truth survey.
PROGRESS_FORM, PROGRESS_PAIN, PROGRESS_SHARE, MIN_DAYS = 0.8, 3, 0.9, 4
EASIER_FORM, EASIER_PAIN, EASIER_TREND = 0.6, 5, -0.15
MAX_REPS = 15
PAUSE_DAYS = 2  # pause an exercise after pain >= 7 on this many days in the week; one day -> lighter


def _ready(ex):
    """This exercise went well enough this week to make it harder."""
    return (ex["days_done"] >= MIN_DAYS and ex["form_avg"] is not None and ex["form_avg"] >= PROGRESS_FORM
            and ex["pain_after_avg"] is not None and ex["pain_after_avg"] <= PROGRESS_PAIN
            and ex["reps_done_share"] >= PROGRESS_SHARE)


def _dose(sets, reps):
    return f"{sets} x {reps}"


def goal_draft(summary):
    """Rule-based draft toward the patient's goal.

    1. Safety first: pause or make easier, whatever the goal.
    2. Exercises below the goal's dose move one step toward it, if this week went well.
    3. At most one exercise the goal needs is added per week, if the whole week went well.
    Returns the new plan, a direction (harder / same / lighter / mixed) and an explanation.
    """
    goal_key = summary.get("goal") if summary.get("goal") in GOALS else DEFAULT_GOAL
    goal = GOALS[goal_key]
    targets = goal["targets"]
    items = []
    for ex in summary["exercises"]:
        sets, reps = (int(x) for x in ex["planned"].split(" x "))
        name = ex["exercise_name"]
        form, pain_avg, pain_max, trend = ex["form_avg"], ex["pain_after_avg"], ex["pain_after_max"], ex["form_trend"]
        target = targets.get(ex["exercise_id"])
        new_sets, new_reps = sets, reps
        if ex["days_done"] == 0:
            action, why = "hold", "Not done this week, so we keep it the same."
        elif ex.get("high_pain_days", 0) >= PAUSE_DAYS:
            action, why, new_sets, new_reps = ("pause", f"Pain reached {pain_max}/10 after it on {ex['high_pain_days']} days. "
                                               "Paused for a week.", 0, 0)
        elif (pain_max is not None and pain_max >= STOP_PAIN) or (form is not None and form < EASIER_FORM) \
                or (pain_avg is not None and pain_avg >= EASIER_PAIN) or (trend is not None and trend <= EASIER_TREND):
            new_reps = adjusted_reps(reps, 0.7)
            bits = []
            if pain_max is not None and pain_max >= STOP_PAIN:
                bits.append(f"pain reached {pain_max}/10 once")
            if pain_avg is not None and pain_avg >= EASIER_PAIN:
                bits.append(f"pain after was {pain_avg}/10")
            if form is not None and form < EASIER_FORM:
                bits.append(f"form was {form}")
            if trend is not None and trend <= EASIER_TREND:
                bits.append("form got worse during the week")
            action, why = "easier", f"Fewer reps because {' and '.join(bits)}."
        elif target and sets * reps < target[0] * target[1]:
            if _ready(ex):
                if reps < target[1]:
                    new_reps = min(target[1], reps + 2)
                else:
                    new_sets = min(target[0], sets + 1)
                action = "progress"
                why = (f"Your goal needs {_dose(*target)}. Form {form} and pain {pain_avg}/10 show you're ready "
                       f"for the next step.")
            else:
                action = "hold"
                why = f"Your goal needs {_dose(*target)}; we'll step up once form is 0.8+ and pain stays 3/10 or less."
        elif target:
            action, why = "hold", "Already at the level your goal needs."
        else:
            action, why = "hold", "Not needed for your goal, but kept as part of your PT's plan."
        items.append({"exercise_id": ex["exercise_id"], "exercise_name": name, "action": action,
                      "before": ex["planned"], "sets": new_sets, "reps": new_reps, "reason": why})

    # Add one missing exercise the goal needs, only after a good week with nothing made easier.
    in_plan = {e["exercise_id"] for e in summary["exercises"]}
    missing = [eid for eid in targets if eid not in in_plan]
    days = summary["days_checked_in"] - summary["days_rest"]
    pain_week = summary["pain_today_avg"]
    week_ok = (days >= MIN_DAYS and pain_week is not None and pain_week <= PROGRESS_PAIN
               and not any(i["action"] in ("easier", "pause") for i in items))
    waiting = []
    if missing and week_ok:
        eid = missing[0]
        t = targets[eid]
        items.append({"exercise_id": eid, "exercise_name": EXERCISES[eid], "action": "add", "before": None,
                      "sets": 2, "reps": min(8, t[1]),
                      "reason": f"New: your goal needs it at {_dose(*t)}. Starting light at 2 x {min(8, t[1])}."})
        waiting = [EXERCISES[e] for e in missing[1:]]
    elif missing:
        waiting = [EXERCISES[e] for e in missing]

    harder = any(i["action"] in ("progress", "add") for i in items)
    lighter = any(i["action"] in ("easier", "pause") for i in items)
    direction = "mixed" if harder and lighter else "harder" if harder else "lighter" if lighter else "same"

    forms = [e["form_avg"] for e in summary["exercises"] if e["form_avg"] is not None]
    week_line = (f"This week you exercised {days} of 7 days, with average pain {pain_week if pain_week is not None else '–'}/10"
                 + (f" and form {round(mean(forms), 2)}." if forms else "."))
    lead = {
        "harder": f"Your goal, \"{goal['label']}\", needs more {goal['focus']} than your current plan, and you're ready for more.",
        "mixed": f"Your goal, \"{goal['label']}\", needs more {goal['focus']}. Some exercises step up, but some need to be lighter first.",
        "lighter": f"Some exercises need to be lighter this week. We'll build toward \"{goal['label']}\" again once they go smoothly.",
        "same": f"Your plan stays the same this week while you build toward \"{goal['label']}\".",
    }[direction]
    changes = [f"{i['exercise_name']}: {i['before'].replace(' x ', ' × ') if i['before'] else 'new'} → "
               f"{'paused' if i['sets'] == 0 else _dose(i['sets'], i['reps']).replace(' x ', ' × ')}"
               for i in items if i["action"] != "hold"]
    explanation = " ".join([lead, week_line] + (["Changes: " + "; ".join(changes) + "."] if changes else []))
    if waiting and direction != "lighter":
        explanation += f" Coming later for your goal: {', '.join(waiting)}."

    note = []
    wk = summary.get("week_post_op")
    if wk is not None and wk < goal["min_week"]:
        note.append(f"Goal \"{goal['label']}\" is usually started from week {goal['min_week']}; patient is in week {wk}. "
                    "Worth discussing before progressing further.")
    if not week_ok and missing:
        note.append("New exercises held back until a week with 4+ sessions, pain 3/10 or less, and nothing made easier.")
    return {"source": "rules", "goal": goal_key, "goal_label": goal["label"], "direction": direction,
            "explanation": explanation, "items": items, "note_for_pt": " ".join(note)}


rule_draft = goal_draft  # older name


LLM_PROMPT = """You draft next week's home-exercise plan for a physical therapist (PT) to review.
The PT decides; you only propose. Work toward the patient's goal, using the goal's target program.
For each exercise in the plan choose: progress, hold, easier, or pause. You may also add ONE exercise
from the goal's target program that is not in the plan (action "add"), only after a good week.
Never progress or add after pain of 7 or more; safety comes before the goal.
Keep reps between 1 and 15 and sets between 0 and 5.
Return JSON: {"direction":"harder|same|lighter|mixed","explanation":"2-3 plain sentences for the patient",
"items":[{"exercise_id","exercise_name","action","before","sets","reps","reason"}],"note_for_pt":"..."}
A rule-based draft is included as a starting point; change it only with a clear reason.

Goal target program:
{targets}

Weekly summary:
{summary}

Rule-based draft:
{rules}
"""


def llm_draft(summary, rules, model=None):
    """Optional LLM draft (OpenAI). Returns None if no API key or the call fails."""
    import os
    if not os.environ.get("OPENAI_API_KEY"):
        return None
    try:
        from openai import OpenAI
        client = OpenAI()
        targets = {EXERCISES[k]: _dose(*v) for k, v in GOALS[rules["goal"]]["targets"].items()}
        resp = client.chat.completions.create(
            model=model or os.environ.get("LLM_MODEL", "gpt-6.2-sol"),
            temperature=0,
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": LLM_PROMPT.replace("{summary}", json.dumps(summary))
                       .replace("{rules}", json.dumps(rules)).replace("{targets}", json.dumps(targets))}],
        )
        draft = json.loads(resp.choices[0].message.content)
    except Exception:
        return None
    return validate_draft(draft, summary, fallback=rules)


def validate_draft(draft, summary, fallback):
    """Code check: known exercises, valid actions and doses, adds only from the goal, no progress after pain >= 7."""
    in_plan = {e["exercise_id"]: e for e in summary["exercises"]}
    goal_ids = set(GOALS[fallback["goal"]]["targets"])
    items = draft.get("items") if isinstance(draft, dict) else None
    if not items or draft.get("direction") not in ("harder", "same", "lighter", "mixed"):
        return fallback
    for it in items:
        eid, action = it.get("exercise_id"), it.get("action")
        if action not in ("progress", "hold", "easier", "pause", "add"):
            return fallback
        if action == "add" and (eid in in_plan or eid not in goal_ids):
            return fallback
        if action != "add" and eid not in in_plan:
            return fallback
        if not (0 <= int(it.get("sets", -1)) <= 5 and 0 <= int(it.get("reps", -1)) <= MAX_REPS):
            return fallback
        if action == "progress" and (in_plan[eid]["pain_after_max"] or 0) >= STOP_PAIN:
            return fallback
        it["exercise_name"] = EXERCISES.get(eid, it.get("exercise_name", eid))
    return {**fallback, "source": "llm", "items": items, "direction": draft["direction"],
            "explanation": draft.get("explanation") or fallback["explanation"],
            "note_for_pt": draft.get("note_for_pt", fallback["note_for_pt"])}


def create_adjustment_request(con, patient_id, week_end=None, use_llm=True):
    """Build the weekly summary and goal-based draft, store it as a pending request for the PT."""
    summary = weekly_summary(con, patient_id, week_end)
    rules = goal_draft(summary)
    draft = (llm_draft(summary, rules) if use_llm else None) or rules
    con.execute("UPDATE adjustment_requests SET status='superseded' WHERE patient_id=? AND status='pending'",
                (patient_id,))
    cur = con.execute(
        "INSERT INTO adjustment_requests (patient_id, week_end, summary_json, draft_json, draft_source, status)"
        " VALUES (?,?,?,?,?, 'pending')",
        (patient_id, summary["week"]["end"], json.dumps(summary), json.dumps(draft), draft["source"]),
    )
    con.commit()
    return {"request_id": cur.lastrowid, "summary": summary, "draft": draft}


def pending_requests(con):
    rows = con.execute(
        "SELECT request_id, patient_id, week_end, draft_source FROM adjustment_requests"
        " WHERE status='pending' ORDER BY request_id").fetchall()
    return [dict(r) for r in rows]


def decide_request(con, request_id, approve, final_items=None, pt_name="PT", pt_note=""):
    """PT approves (optionally edited) or rejects. Approval replaces the patient's plan."""
    req = con.execute("SELECT * FROM adjustment_requests WHERE request_id=?", (request_id,)).fetchone()
    if req is None or req["status"] != "pending":
        raise ValueError("No pending request with that id.")
    draft_items = json.loads(req["draft_json"])["items"]
    final_items = final_items if final_items is not None else draft_items
    edited = int(json.dumps(final_items, sort_keys=True) != json.dumps(draft_items, sort_keys=True))
    now = dt.datetime.now().isoformat(timespec="seconds")
    if approve:
        new_plan = [{"exercise_id": i["exercise_id"], "exercise_name": i["exercise_name"],
                     "sets": i["sets"], "reps": i["reps"]} for i in final_items if int(i["sets"]) > 0]
        week_end = dt.date.fromisoformat(req["week_end"])
        set_plan(con, req["patient_id"], new_plan, approved_by=pt_name,
                 start_date=(week_end + dt.timedelta(days=1)).isoformat())
    con.execute(
        "UPDATE adjustment_requests SET status=?, final_json=?, pt_edited=?, pt_note=?, decided_at=?"
        " WHERE request_id=?",
        ("approved" if approve else "rejected", json.dumps(final_items), edited, pt_note, now, request_id),
    )
    con.commit()
    return {"request_id": request_id, "status": "approved" if approve else "rejected", "pt_edited": bool(edited)}


# ---------- Intake (tab 1) ----------

def intake_summary(a):
    """Short summary for the PT. Rule-based placeholder until the LLM summary is added."""
    def j(x):
        return ", ".join(x) if isinstance(x, list) and x else (x or "not given")
    lines = [
        f"{a.get('age', '?')} y/o, {a.get('daily_activity', 'activity not given')}; "
        f"walking aid: {a.get('walking_aid', 'not given')}.",
        f"Knee pain at {j(a.get('pain_locations'))}, {j(a.get('pain_type'))}, for {a.get('duration', '?')}; "
        f"{a.get('pain_pattern', '?')}. Now {a.get('pain_now', '?')}/10, worst {a.get('pain_worst', '?')}/10.",
        f"Worse with: {j(a.get('worse_with'))}. Better with: {a.get('better_with') or 'not given'}. "
        f"Mostly in the {j(a.get('time_of_day'))}.",
    ]
    if a.get("numbness") == "yes":
        lines.append("Reports numbness: check in person.")
    if a.get("surgery") and a.get("surgery") != "none":
        lines.append(f"Surgery: {a['surgery']} on {a.get('surgery_date') or 'date not given'}.")
    goal = GOALS.get(a.get("goal"), {}).get("label", a.get("goal") or "not given")
    lines.append(f"Goal: {goal}. Home: {j(a.get('home_equipment'))}; "
                 f"can get to the floor: {a.get('can_get_to_floor', '?')}.")
    return "\n".join(lines)


def save_intake(con, patient_id, name, answers):
    """Store intake answers, create or update the patient, return the summary for the PT."""
    old = con.execute("SELECT * FROM patients WHERE patient_id=?", (patient_id,)).fetchone()
    add_patient(con, patient_id, name or (old["name"] if old else ""),
                answers.get("surgery_date") or (old["surgery_date"] if old else ""),
                answers.get("side") or (old["operated_side"] if old else ""),
                answers.get("goal") or (old["goal"] if old else ""))
    summary = intake_summary(answers)
    con.execute("INSERT INTO intakes (patient_id, created_at, answers_json, summary) VALUES (?,?,?,?)",
                (patient_id, dt.datetime.now().isoformat(timespec="seconds"), json.dumps(answers), summary))
    con.commit()
    return summary


def latest_intake(con, patient_id):
    r = con.execute("SELECT * FROM intakes WHERE patient_id=? ORDER BY intake_id DESC LIMIT 1",
                    (patient_id,)).fetchone()
    if r is None:
        return None
    return {"created_at": r["created_at"], "answers": json.loads(r["answers_json"]), "summary": r["summary"]}


def set_goal(con, patient_id, goal):
    if goal not in GOALS:
        raise ValueError(f"Unknown goal '{goal}'. Choose one of: {', '.join(GOALS)}")
    con.execute("UPDATE patients SET goal=? WHERE patient_id=?", (goal, patient_id))
    con.commit()


# ---------- Plan views (tab 3) ----------

def latest_request(con, patient_id, status=None):
    sql = "SELECT * FROM adjustment_requests WHERE patient_id=?"
    args = [patient_id]
    if status:
        sql += " AND status=?"
        args.append(status)
    r = con.execute(sql + " ORDER BY request_id DESC LIMIT 1", args).fetchone()
    if r is None:
        return None
    return {"request_id": r["request_id"], "week_end": r["week_end"], "status": r["status"],
            "draft_source": r["draft_source"], "draft": json.loads(r["draft_json"]),
            "summary": json.loads(r["summary_json"]),
            "final": json.loads(r["final_json"]) if r["final_json"] else None,
            "pt_note": r["pt_note"], "decided_at": r["decided_at"]}


def plan_change_for_patient(con, patient_id):
    """What the patient sees after the PT approves: old vs new per exercise, with reasons."""
    req = latest_request(con, patient_id, "approved")
    if req is None:
        return None
    old = {e["exercise_id"]: e for e in req["summary"]["exercises"]}
    reasons = {i["exercise_id"]: i.get("reason", "") for i in req["draft"]["items"]}
    actions = {i["exercise_id"]: i.get("action") for i in req["final"]}
    changes = []
    for it in req["final"]:
        prev = old.get(it["exercise_id"])
        before = prev["planned"] if prev else None
        after = f"{it['sets']} x {it['reps']}" if int(it["sets"]) > 0 else None
        kind = "new" if before is None else "paused" if after is None else "same" if before == after else "changed"
        changes.append({"exercise_id": it["exercise_id"], "exercise_name": it["exercise_name"],
                        "kind": kind, "action": actions.get(it["exercise_id"]), "before": before, "after": after, "reason": reasons.get(it["exercise_id"], "")})
    return {"approved_at": req["decided_at"], "pt_note": req["pt_note"], "changes": changes,
            "direction": req["draft"].get("direction"), "explanation": req["draft"].get("explanation", ""),
            "goal_label": req["draft"].get("goal_label", "")}


# ---------- Mock data for testing ----------

DEMO_PLAN = [
    {"exercise_id": "heel_slide", "exercise_name": "Heel slide", "sets": 3, "reps": 10},
    {"exercise_id": "sit_to_stand", "exercise_name": "Sit-to-stand", "sets": 3, "reps": 10},
    {"exercise_id": "mini_squat", "exercise_name": "Mini squat", "sets": 3, "reps": 10},
]


def load_demo_week(con, patient_id="u_demo", week_end="2026-11-22"):
    """Seed one patient with a PT plan and 7 days of sessions (pain rises after mini squats)."""
    end = dt.date.fromisoformat(week_end)
    start = (end - dt.timedelta(days=6)).isoformat()
    for table in ("exercise_logs", "checkins", "alerts"):  # loading twice must not double the data
        con.execute(f"DELETE FROM {table} WHERE patient_id=? AND date BETWEEN ? AND ?", (patient_id, start, week_end))
    con.execute("UPDATE adjustment_requests SET status='superseded' WHERE patient_id=? AND status='pending'", (patient_id,))
    con.commit()
    if con.execute("SELECT 1 FROM patients WHERE patient_id=?", (patient_id,)).fetchone() is None:
        add_patient(con, patient_id, "Walter M.", "2026-10-28", "left", "stairs")
    if not current_plan(con, patient_id):  # keep the PT's plan if there is one
        set_plan(con, patient_id, DEMO_PLAN, approved_by="Dana Reyes, DPT",
                 start_date=(end - dt.timedelta(days=6)).isoformat())
    pains = [2, 2, 3, 3, 2, 4, 2]
    for i, pain in enumerate(pains):
        date = (end - dt.timedelta(days=6 - i)).isoformat()
        if i == 3:  # one missed day
            continue
        today = daily_checkin(con, patient_id, pain, date, pain_location="front" if pain >= 4 else "none")
        for ex in today["exercises"]:
            n = ex["sets"] * ex["target_reps"]
            squat = ex["exercise_id"] == "mini_squat"
            log_exercise(con, today["checkin_id"], {
                "exercise_id": ex["exercise_id"],
                "sets_done": ex["sets"],
                "reps_done": n - (2 if squat else 0),
                "form_scores": [0.9 - (0.05 * i if squat else 0.0)] * max(n, 1),
                "error_counts": {"trunk_lean": i} if squat else {},
                "pain_after": min(10, pain + (2 if squat else 0)),
            })


## 3. `test_server.py`: HTTP API (the UI and the detection part call these endpoints)

In [ ]:
%%writefile /content/rehab_node/test_server.py
"""Knee rehab app skeleton (3 tabs) + the detection -> node test bench. Standard library only.

Tabs: 1 Intake (patient answers, PT reads summary and enters the plan)
      2 Daily exercise (pain check, detection result -> node; camera to be added)
      3 Plan (patient sees plan and changes; PT reviews the weekly draft)

Run:  python test_server.py      then open http://127.0.0.1:8000
      (Colab: see rehab_colab.ipynb)

Endpoint the detection node calls after each exercise:
  POST /api/result
  {"checkin_id": 1, "exercise_id": "mini_squat", "sets_done": 3, "reps_done": 21,
   "form_scores": [0.9, 0.8, ...], "error_counts": {"trunk_lean": 2}, "pain_after": 4}
"""
import json
import os
import webbrowser
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

import rehab_node as rn

HERE = os.path.dirname(os.path.abspath(__file__))
DB_PATH = os.environ.get("REHAB_DB", os.path.join(HERE, "rehab_test.db"))  # Colab: set to a Drive path to keep data
con = rn.connect(DB_PATH)
PORT = int(os.environ.get("PORT", 8000))
ALLOW_ORIGIN = os.environ.get("ALLOW_ORIGIN", "*")  # CORS, so a separate frontend (Figma Make/React) can call the API


def check_detection(body):
    """Return a list of problems with a detection result; empty list = valid."""
    errs = []
    for key in ("checkin_id", "exercise_id", "sets_done", "reps_done", "form_scores"):
        if key not in body:
            errs.append(f"missing '{key}'")
    for key in ("checkin_id", "sets_done", "reps_done"):
        if key in body and (not isinstance(body[key], int) or body[key] < 0):
            errs.append(f"'{key}' must be a whole number >= 0")
    fs = body.get("form_scores")
    if fs is not None:
        if not isinstance(fs, list) or not all(isinstance(x, (int, float)) and 0 <= x <= 1 for x in fs):
            errs.append("'form_scores' must be a list of numbers from 0 to 1, one per rep")
        elif "reps_done" in body and isinstance(body["reps_done"], int) and len(fs) != body["reps_done"]:
            errs.append(f"'form_scores' has {len(fs)} values but reps_done is {body['reps_done']} (expected one per rep)")
    ec = body.get("error_counts", {})
    if not isinstance(ec, dict) or not all(isinstance(v, int) and v >= 0 for v in ec.values()):
        errs.append("'error_counts' must be an object of error type -> count")
    pa = body.get("pain_after")
    if pa is not None and (not isinstance(pa, int) or not 0 <= pa <= 10):
        errs.append("'pain_after' must be a whole number 0-10")
    return errs


def rows(sql, args=()):
    return [dict(r) for r in con.execute(sql, args).fetchall()]


class Handler(BaseHTTPRequestHandler):
    def _send(self, code, payload, ctype="application/json"):
        data = payload if isinstance(payload, bytes) else json.dumps(payload).encode()
        self.send_response(code)
        self._cors()
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def _cors(self):
        self.send_header("Access-Control-Allow-Origin", ALLOW_ORIGIN)
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")

    def do_OPTIONS(self):  # CORS preflight
        self.send_response(204)
        self._cors()
        self.end_headers()

    def _body(self):
        n = int(self.headers.get("Content-Length") or 0)
        return json.loads(self.rfile.read(n) or b"{}")

    def log_message(self, *args):  # keep the console quiet
        pass

    def do_GET(self):
        url = urlparse(self.path)
        q = {k: v[0] for k, v in parse_qs(url.query).items()}
        if url.path in ("/", "/index.html"):
            with open(os.path.join(HERE, "test_page.html"), "rb") as f:
                return self._send(200, f.read(), "text/html; charset=utf-8")
        pid = q.get("patient_id", "")
        if url.path == "/api/exercises":
            return self._send(200, rn.EXERCISES)
        if url.path == "/api/goals":
            return self._send(200, [{"key": k, "label": g["label"]} for k, g in rn.GOALS.items()])
        if url.path == "/api/intake":
            return self._send(200, {"intake": rn.latest_intake(con, pid)})
        if url.path == "/api/plan":
            p = con.execute("SELECT * FROM patients WHERE patient_id=?", (pid,)).fetchone()
            return self._send(200, {"patient": dict(p) if p else None, "plan": rn.current_plan(con, pid)})
        if url.path == "/api/adjustment":
            return self._send(200, {"pending": rn.latest_request(con, pid, "pending"),
                                    "plan": rn.current_plan(con, pid)})
        if url.path == "/api/myplan":
            p = con.execute("SELECT * FROM patients WHERE patient_id=?", (pid,)).fetchone()
            return self._send(200, {"patient": dict(p) if p else None, "plan": rn.current_plan(con, pid),
                                    "change": rn.plan_change_for_patient(con, pid)})
        if url.path == "/api/db":
            pid = q.get("patient_id", "")
            return self._send(200, {
                "logs": rows("SELECT log_id, date, exercise_id, planned_sets, planned_reps, target_reps, sets_done,"
                             " reps_done, form_score, errors_json, pain_after FROM exercise_logs"
                             " WHERE patient_id=? ORDER BY log_id DESC LIMIT 50", (pid,)),
                "alerts": rows("SELECT date, tier, reason FROM alerts WHERE patient_id=? ORDER BY alert_id DESC LIMIT 20",
                               (pid,)),
            })
        self._send(404, {"error": "not found"})

    def do_POST(self):
        try:
            body = self._body()
        except json.JSONDecodeError as e:
            return self._send(400, {"ok": False, "errors": [f"invalid JSON: {e}"]})
        path = urlparse(self.path).path
        try:
            if path == "/api/test-patient":
                pid = body.get("patient_id") or "u_test"
                rn.add_patient(con, pid, "Test patient", "2026-10-28", "left", "Climb stairs at home")
                rn.set_plan(con, pid, rn.DEMO_PLAN, approved_by="Test PT")
                return self._send(200, {"ok": True, "plan": rn.current_plan(con, pid)})
            if path == "/api/intake":
                summary = rn.save_intake(con, body["patient_id"], body.get("name", ""), body["answers"])
                return self._send(200, {"ok": True, "summary": summary})
            if path == "/api/plan":
                items = [{"exercise_id": i["exercise_id"], "exercise_name": rn.EXERCISES[i["exercise_id"]],
                          "sets": int(i["sets"]), "reps": int(i["reps"])} for i in body["items"]]
                if not items:
                    raise ValueError("Add at least one exercise.")
                if con.execute("SELECT 1 FROM patients WHERE patient_id=?", (body["patient_id"],)).fetchone() is None:
                    raise ValueError("No patient with that ID. The patient fills in the intake first.")
                rn.set_plan(con, body["patient_id"], items, approved_by=body.get("pt_name") or "PT")
                return self._send(200, {"ok": True, "plan": rn.current_plan(con, body["patient_id"])})
            if path == "/api/goal":
                pid = body["patient_id"]
                rn.set_goal(con, pid, body.get("goal", ""))
                # A new goal means a new draft for the PT, if there are sessions to base it on.
                last = con.execute("SELECT MAX(date) FROM checkins WHERE patient_id=?", (pid,)).fetchone()[0]
                if last and rn.current_plan(con, pid):
                    out = rn.create_adjustment_request(con, pid, last)
                    return self._send(200, {"ok": True, "drafted": True, "request_id": out["request_id"]})
                return self._send(200, {"ok": True, "drafted": False})
            if path == "/api/demo-week":
                pid = body.get("patient_id") or "u_demo"
                rn.load_demo_week(con, pid)
                return self._send(200, {"ok": True, "patient_id": pid})
            if path == "/api/adjustment/draft":
                week_end = body.get("week_end") or (con.execute(  # default: last day the patient checked in
                    "SELECT MAX(date) FROM checkins WHERE patient_id=?", (body["patient_id"],)).fetchone()[0])
                out = rn.create_adjustment_request(con, body["patient_id"], week_end)
                return self._send(200, {"ok": True, "request_id": out["request_id"]})
            if path == "/api/adjustment/decide":
                res = rn.decide_request(con, int(body["request_id"]), bool(body["approve"]), body.get("items"),
                                        body.get("pt_name") or "PT", body.get("note", ""))
                return self._send(200, {"ok": True, **res})
            if path == "/api/checkin":
                today = rn.daily_checkin(con, body["patient_id"], int(body["pain_today"]),
                                         body.get("date") or None, body.get("pain_location", "none"))
                return self._send(200, {"ok": True, **today})
            if path == "/api/result":
                errs = check_detection(body)
                if errs:
                    return self._send(422, {"ok": False, "errors": errs})
                rn.log_exercise(con, body["checkin_id"], body)
                saved = rows("SELECT * FROM exercise_logs ORDER BY log_id DESC LIMIT 1")[0]
                alert = body.get("pain_after") is not None and body["pain_after"] >= rn.STOP_PAIN
                return self._send(200, {"ok": True, "saved": saved, "alert_sent": alert})
        except (KeyError, ValueError) as e:
            return self._send(400, {"ok": False, "errors": [str(e)]})
        self._send(404, {"error": "not found"})


def start(host="127.0.0.1", port=PORT):
    """Start the server in a background thread (used by the Colab notebook). Returns the server."""
    import threading
    server = ThreadingHTTPServer((host, port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server


if __name__ == "__main__":
    server = ThreadingHTTPServer(("127.0.0.1", PORT), Handler)
    url = f"http://127.0.0.1:{PORT}"
    print(f"Test bench running at {url}  (Ctrl+C to stop)")
    try:
        webbrowser.open(url)
    except Exception:
        pass
    server.serve_forever()


## 4. `test_page.html`: the 3-tab interface (Intake / Daily exercise / Plan)

In [ ]:
%%writefile /content/rehab_node/test_page.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Knee rehab app (prototype)</title>
<style>
  :root{
    --bg:#f5f3ee; --card:#ffffff; --ink:#1f1d1a; --muted:#6b675f; --line:#e4e0d6;
    --accent:#2f6f5e; --accent-soft:#e3efe9; --warn:#b26a00; --warn-soft:#fbefd9;
    --bad:#b23a2e; --bad-soft:#f8e3e0; --pt:#5b4bb0; --pt-soft:#ebe8f8; --code:#f8f6f1; --radius:14px;
    --mono: ui-monospace, "Cascadia Code", Consolas, monospace;
  }
  @media (prefers-color-scheme: dark){
    :root{ --bg:#141412; --card:#1e1d1a; --ink:#ece9e2; --muted:#a29d92; --line:#33312c;
      --accent:#6fc3a8; --accent-soft:#1f3830; --warn:#e7a948; --warn-soft:#3a2d16;
      --bad:#f08a7e; --bad-soft:#3d201c; --pt:#ada2f0; --pt-soft:#2a2640; --code:#171614; }
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.5 "Segoe UI",system-ui,sans-serif}
  header{padding:20px 32px 0;display:flex;align-items:center;gap:18px;flex-wrap:wrap}
  h1{font-size:21px;margin:0;font-weight:650;letter-spacing:-.01em}
  .pidbox{display:flex;align-items:center;gap:8px;margin-left:auto}
  .pidbox input{width:140px}
  nav{display:flex;gap:6px;padding:16px 32px 0;border-bottom:1px solid var(--line)}
  nav button{background:none;border:0;border-bottom:2px solid transparent;border-radius:0;padding:10px 14px;color:var(--muted);font-weight:600}
  nav button.on{color:var(--ink);border-bottom-color:var(--accent)}
  .tab{display:none}
  .tab.on{display:grid}
  main{grid-template-columns:1fr 1fr;gap:18px;padding:18px 32px 32px}
  main.wide{grid-template-columns:minmax(280px,1fr) minmax(360px,1.35fr)}
  .full{grid-column:1/-1}
  @media (max-width:900px){main,main.wide{grid-template-columns:1fr;padding:14px}}
  .card{background:var(--card);border:1px solid var(--line);border-radius:var(--radius);padding:20px 22px;align-self:start}
  .card.ptcard{border-top:4px solid var(--pt)}
  .card.patcard{border-top:4px solid var(--accent)}
  .who{font-size:12px;font-weight:700;letter-spacing:.06em;text-transform:uppercase;margin-bottom:4px}
  .ptcard .who{color:var(--pt)} .patcard .who{color:var(--accent)}
  .step{display:flex;align-items:center;gap:10px;margin-bottom:14px}
  .num{width:26px;height:26px;border-radius:50%;background:var(--accent);color:#fff;display:grid;place-items:center;font-weight:650;font-size:13px}
  h2{font-size:17px;margin:0 0 10px;font-weight:650}
  .step h2{margin:0;font-size:16px}
  h3{font-size:14px;margin:18px 0 6px;font-weight:650}
  label{display:block;font-size:13px;color:var(--muted);margin:12px 0 5px}
  input[type=text],input[type=number],input[type=date],select,textarea{width:100%;padding:9px 11px;border:1px solid var(--line);border-radius:9px;background:var(--card);color:var(--ink);font:inherit}
  textarea{font:13px/1.55 var(--mono);background:var(--code);min-height:200px;resize:vertical}
  .grid2{display:grid;grid-template-columns:1fr 1fr;gap:0 14px}
  .row{display:flex;gap:10px;align-items:center}
  .row > *{flex:1}
  .chips{display:flex;flex-wrap:wrap;gap:6px}
  .chips label{margin:0;display:inline-flex;align-items:center;gap:6px;padding:6px 11px;border:1px solid var(--line);border-radius:999px;color:var(--ink);font-size:13.5px;cursor:pointer}
  .chips input{accent-color:var(--accent);margin:0}
  .slider{display:flex;align-items:center;gap:12px}
  .slider input{flex:1;accent-color:var(--accent)}
  .val{min-width:44px;text-align:center;font-weight:650;padding:3px 8px;border-radius:8px;background:var(--accent-soft);color:var(--accent)}
  button{font:inherit;font-weight:600;border:0;border-radius:10px;padding:10px 16px;cursor:pointer}
  .primary{background:var(--accent);color:#fff}
  .ptbtn{background:var(--pt);color:#fff}
  .ghost{background:transparent;border:1px solid var(--line);color:var(--ink)}
  .small{padding:6px 10px;font-size:13px}
  button:disabled{opacity:.45;cursor:not-allowed}
  .actions{display:flex;gap:10px;margin-top:16px;flex-wrap:wrap}
  .banner{margin-top:14px;padding:10px 14px;border-radius:10px;font-weight:600}
  .ok{background:var(--accent-soft);color:var(--accent)}
  .warnb{background:var(--warn-soft);color:var(--warn)}
  .badb{background:var(--bad-soft);color:var(--bad)}
  .ptb{background:var(--pt-soft);color:var(--pt)}
  .plan{list-style:none;padding:0;margin:12px 0 0;display:grid;gap:8px}
  .plan li{display:flex;justify-content:space-between;align-items:center;gap:10px;padding:11px 14px;border:1px solid var(--line);border-radius:10px}
  .plan.pick li{cursor:pointer}
  .plan li.sel{border-color:var(--accent);box-shadow:0 0 0 2px var(--accent-soft)}
  .plan small{color:var(--muted)}
  .dose{white-space:nowrap}
  .dose s{color:var(--muted);margin-right:6px}
  .io{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-top:16px}
  @media (max-width:700px){.io,.grid2{grid-template-columns:1fr}}
  pre{margin:0;font:12px/1.5 var(--mono);background:var(--code);border:1px solid var(--line);border-radius:10px;padding:12px;overflow:auto;max-height:260px;white-space:pre-wrap}
  pre.summary{font:14px/1.6 "Segoe UI",system-ui,sans-serif;max-height:none}
  .tag{font-size:12px;font-weight:650;padding:2px 8px;border-radius:6px;white-space:nowrap}
  table{width:100%;border-collapse:collapse;font-size:14px}
  th{text-align:left;color:var(--muted);font-weight:600;font-size:12px;text-transform:uppercase;letter-spacing:.04em;padding:8px 10px;border-bottom:1px solid var(--line)}
  td{padding:9px 10px;border-bottom:1px solid var(--line);vertical-align:top}
  th.n,td.n{text-align:right;font-variant-numeric:tabular-nums}
  td input[type=number]{width:52px;padding:5px 6px}
  .prop{display:flex;align-items:center;gap:4px;white-space:nowrap;margin-top:6px}
  td.n{white-space:nowrap}
  td select{width:auto;padding:5px 7px}
  .reason{color:var(--muted);font-size:13px}
  .empty{color:var(--muted);padding:10px 0}
  .alerts{margin-top:14px;display:grid;gap:6px}
  .hint{font-size:13px;color:var(--muted);margin-top:8px}
  code{font-family:var(--mono);font-size:12.5px;background:var(--code);padding:1px 5px;border-radius:5px}
  .camera{border:2px dashed var(--line);border-radius:12px;aspect-ratio:16/9;display:grid;place-items:center;color:var(--muted);text-align:center;padding:20px}
  .dir{display:inline-block;font-weight:700;font-size:13px;padding:3px 10px;border-radius:999px;margin-right:8px}
  .expl{background:var(--code);border:1px solid var(--line);border-radius:10px;padding:12px 14px;margin:10px 0;line-height:1.6}
  .edit-on .ro{display:none} .ro-edit{display:none} .edit-on .ro-edit{display:flex}
  .stats{display:flex;gap:10px;flex-wrap:wrap;margin:6px 0 4px}
  .stat{border:1px solid var(--line);border-radius:10px;padding:8px 12px;min-width:110px}
  .stat b{display:block;font-size:18px}
  .stat span{font-size:12px;color:var(--muted)}
</style>
</head>
<body>
<header>
  <h1>Knee rehab app <span style="color:var(--muted);font-weight:400">· prototype</span></h1>
  <div class="pidbox"><label for="pid" style="margin:0">Patient ID</label><input type="text" id="pid" value="u_test"></div>
</header>
<nav>
  <button data-tab="t1" class="on">1 · Intake</button>
  <button data-tab="t2">2 · Daily exercise</button>
  <button data-tab="t3">3 · Plan</button>
</nav>

<!-- ============ TAB 1: INTAKE ============ -->
<main class="tab on" id="t1">
  <section class="card patcard">
    <div class="who">Patient</div>
    <h2>Tell us about your knee</h2>
    <div class="grid2">
      <div><label for="i_name">Name</label><input type="text" id="i_name" value="Walter M."></div>
      <div><label for="i_age">Age</label><input type="number" id="i_age" value="76" min="18" max="110"></div>
      <div><label for="i_act">Most of your day is spent…</label>
        <select id="i_act"><option>sitting or desk work</option><option>standing</option><option>walking</option><option>a mix of these</option></select></div>
      <div><label for="i_aid">Walking aid</label><select id="i_aid"><option>none</option><option selected>cane</option><option>walker</option></select></div>
    </div>
    <label>Where does it hurt?</label>
    <div class="chips" data-name="pain_locations">
      <label><input type="checkbox" value="front" checked>Front</label><label><input type="checkbox" value="back">Back</label>
      <label><input type="checkbox" value="inner side">Inner side</label><label><input type="checkbox" value="outer side">Outer side</label>
      <label><input type="checkbox" value="above the knee">Above the knee</label><label><input type="checkbox" value="below the knee">Below the knee</label>
      <label><input type="checkbox" value="whole knee">Whole knee</label></div>
    <label>What does it feel like?</label>
    <div class="chips" data-name="pain_type">
      <label><input type="checkbox" value="dull ache" checked>Dull ache</label><label><input type="checkbox" value="sharp">Sharp</label>
      <label><input type="checkbox" value="burning">Burning</label><label><input type="checkbox" value="stiff">Stiff</label></div>
    <div class="grid2">
      <div><label for="i_dur">How long?</label>
        <select id="i_dur"><option>under 2 weeks</option><option selected>2–6 weeks</option><option>6 weeks–3 months</option><option>over 3 months</option></select></div>
      <div><label for="i_pat">Is it…</label><select id="i_pat"><option>there all the time</option><option selected>on and off</option></select></div>
    </div>
    <label>When is it worst?</label>
    <div class="chips" data-name="time_of_day">
      <label><input type="checkbox" value="morning">Morning</label><label><input type="checkbox" value="after activity" checked>After activity</label>
      <label><input type="checkbox" value="evening">Evening</label><label><input type="checkbox" value="night">Night</label></div>
    <label>What makes it worse?</label>
    <div class="chips" data-name="worse_with">
      <label><input type="checkbox" value="stairs" checked>Stairs</label><label><input type="checkbox" value="squatting or kneeling">Squatting or kneeling</label>
      <label><input type="checkbox" value="sitting a long time">Sitting a long time</label><label><input type="checkbox" value="walking">Walking</label>
      <label><input type="checkbox" value="getting up from a chair" checked>Getting up from a chair</label></div>
    <label for="i_better">What makes it better?</label><input type="text" id="i_better" value="Rest and ice">
    <div class="grid2">
      <div><label>Pain right now</label><div class="slider"><input type="range" id="i_now" min="0" max="10" value="4"><span class="val" id="i_nowv">4</span></div></div>
      <div><label>Worst pain this week</label><div class="slider"><input type="range" id="i_worst" min="0" max="10" value="6"><span class="val" id="i_worstv">6</span></div></div>
      <div><label for="i_numb">Any numbness or tingling?</label><select id="i_numb"><option>no</option><option>yes</option></select></div>
      <div><label for="i_side">Which knee?</label><select id="i_side"><option>left</option><option>right</option></select></div>
      <div><label for="i_surg">Knee surgery?</label><select id="i_surg"><option>none</option><option selected>total knee replacement</option><option>ACL repair</option><option>other</option></select></div>
      <div><label for="i_sdate">Surgery date</label><input type="date" id="i_sdate" value="2026-10-28"></div>
    </div>
    <label for="i_goal">What do you most want to get back to?</label><select id="i_goal" class="goalsel"></select>
    <label>What do you have at home?</label>
    <div class="chips" data-name="home_equipment">
      <label><input type="checkbox" value="sturdy chair" checked>Sturdy chair</label><label><input type="checkbox" value="step or stairs" checked>Step or stairs</label>
      <label><input type="checkbox" value="bed" checked>Bed</label><label><input type="checkbox" value="floor space">Floor space</label>
      <label><input type="checkbox" value="exercise bike">Exercise bike</label></div>
    <label for="i_floor">Can you get down to the floor and back up safely?</label><select id="i_floor"><option>no</option><option>yes</option></select>
    <div class="actions"><button class="primary" id="i_send">Send to my PT</button></div>
    <div id="i_msg"></div>
  </section>

  <section class="card ptcard">
    <div class="who">Physical therapist</div>
    <h2>Before the first visit</h2>
    <div class="row"><h3 style="margin:0">Intake summary</h3><div style="flex:0 0 auto"><button class="ghost small" id="pt_refresh1">Refresh</button></div></div>
    <pre class="summary" id="pt_summary" style="margin-top:8px">No intake yet for this patient.</pre>
    <h3>Initial plan, after the in-person assessment</h3>
    <table><thead><tr><th>Exercise</th><th class="n">Sets</th><th class="n">Reps</th><th></th></tr></thead><tbody id="pt_rows"></tbody></table>
    <div class="actions" style="margin-top:10px"><button class="ghost small" id="pt_add">Add exercise</button></div>
    <label for="pt_name">PT name</label><input type="text" id="pt_name" value="Dana Reyes, DPT">
    <div class="actions"><button class="ptbtn" id="pt_save">Save plan</button></div>
    <div id="pt_msg"></div>
  </section>
</main>

<!-- ============ TAB 2: DAILY EXERCISE ============ -->
<main class="tab wide" id="t2">
  <section class="card">
    <div class="step"><span class="num">1</span><h2>Start today's session</h2></div>
    <label for="loc">Where does your knee hurt today?</label>
    <select id="loc">
      <option value="none">No pain</option><option value="front">Front</option><option value="back">Back</option>
      <option value="inner">Inner side</option><option value="outer">Outer side</option><option value="whole knee">Whole knee</option>
    </select>
    <label>How much does it hurt today? (0 = none, 10 = worst)</label>
    <div class="slider"><input type="range" id="pain" min="0" max="10" value="2"><span class="val" id="painv">2</span></div>
    <div class="actions"><button class="primary" id="start">Start session</button><button class="ghost" id="seed">Load test patient</button></div>
    <div id="banner"></div>
    <ul class="plan pick" id="plan"></ul>
    <p class="hint" id="planhint"></p>
  </section>

  <section class="card">
    <div class="step"><span class="num">2</span><h2>Camera and detection</h2></div>
    <div class="camera">Camera view goes here<br><small>Pose model, live cues and rep counter will be added. For now, simulate the detection output below.</small></div>
    <label for="json">Detection JSON (sent to <code>POST /api/result</code>)</label>
    <textarea id="json" spellcheck="false" placeholder="Start a session and pick an exercise first."></textarea>
    <label>Pain after this exercise (asked by the app, added to the request)</label>
    <div class="slider"><input type="range" id="after" min="0" max="10" value="3"><span class="val" id="afterv">3</span></div>
    <div class="actions">
      <button class="ghost" id="sim" disabled>Simulate detection</button>
      <button class="ghost" id="bad" disabled>Simulate a broken result</button>
      <button class="primary" id="send" disabled>Send to node</button>
    </div>
    <div class="io">
      <div><label>Request sent</label><pre id="req">—</pre></div>
      <div><label>Node response <span id="status"></span></label><pre id="res">—</pre></div>
    </div>
  </section>

  <section class="card full">
    <div class="step"><span class="num">3</span><h2>What the node stored</h2></div>
    <div id="table"></div>
    <div class="alerts" id="alerts"></div>
  </section>
</main>

<!-- ============ TAB 3: PLAN ============ -->
<main class="tab" id="t3">
  <section class="card patcard">
    <div class="who">Patient</div>
    <h2>My plan</h2>
    <div id="my_change"></div>
    <h3>This week's exercises</h3>
    <ul class="plan" id="my_plan"></ul>
    <h3>My goal</h3>
    <div class="row"><select id="my_goal" class="goalsel"></select><div style="flex:0 0 auto"><button class="primary" id="my_goal_save">Save goal</button></div></div>
    <p class="hint">Changing your goal makes a new plan for your PT to check.</p>
    <div id="my_msg"></div>
  </section>

  <section class="card ptcard">
    <div class="who">Physical therapist</div>
    <h2>Weekly plan review</h2>
    <p class="hint" style="margin-top:0">In the real app the draft is made automatically at the end of each week. For testing, make it here.</p>
    <div class="actions" style="margin-top:4px">
      <button class="ghost small" id="rv_demo">Load a demo week for this patient</button>
      <button class="ghost small" id="rv_draft">Make this week's draft</button>
      <button class="ghost small" id="rv_refresh">Refresh</button>
    </div>
    <div id="rv_body"><p class="empty">No draft waiting for review.</p></div>
    <div id="rv_msg"></div>
  </section>
</main>

<script>
const $ = id => document.getElementById(id);
const pid = () => $("pid").value.trim();
const pretty = o => JSON.stringify(o, null, 2).replace(/\[\s+([\d.,\s-]+?)\s+\]/g, (m, inner) => "[" + inner.replace(/\s+/g, " ").trim() + "]");
const esc = s => String(s ?? "").replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const banner = (cls, html) => `<div class="banner ${cls}">${html}</div>`;
const api = async (path, body) => {
  const r = await fetch(path, body ? {method:"POST", headers:{"Content-Type":"application/json"}, body:JSON.stringify(body)} : {});
  return {status:r.status, data:await r.json()};
};
const errText = d => (d.errors || [d.error || "Something went wrong"]).map(esc).join("<br>");
let EX = {}, GOALS = [];

// ---------- tabs ----------
document.querySelectorAll("nav button").forEach(b => b.onclick = () => {
  document.querySelectorAll("nav button").forEach(x => x.classList.toggle("on", x === b));
  document.querySelectorAll(".tab").forEach(t => t.classList.toggle("on", t.id === b.dataset.tab));
  refreshTab(b.dataset.tab);
});
function refreshTab(t){ if (t === "t1") loadIntake(); if (t === "t2") loadDb(); if (t === "t3"){ loadMyPlan(); loadReview(); } }
$("pid").onchange = () => refreshTab(document.querySelector("nav button.on").dataset.tab);
const bindVal = (s, v) => $(s).oninput = e => $(v).textContent = e.target.value;
bindVal("i_now","i_nowv"); bindVal("i_worst","i_worstv"); bindVal("pain","painv"); bindVal("after","afterv");

// ---------- TAB 1: intake (patient) ----------
const chips = name => [...document.querySelectorAll(`.chips[data-name="${name}"] input:checked`)].map(i => i.value);
$("i_send").onclick = async () => {
  const answers = {
    age:+$("i_age").value, daily_activity:"mostly " + $("i_act").value, walking_aid:$("i_aid").value,
    pain_locations:chips("pain_locations"), pain_type:chips("pain_type"), duration:$("i_dur").value,
    pain_pattern:$("i_pat").value, time_of_day:chips("time_of_day"), worse_with:chips("worse_with"),
    better_with:$("i_better").value, pain_now:+$("i_now").value, pain_worst:+$("i_worst").value,
    numbness:$("i_numb").value, side:$("i_side").value, surgery:$("i_surg").value,
    surgery_date:$("i_surg").value === "none" ? "" : $("i_sdate").value, goal:$("i_goal").value,
    home_equipment:chips("home_equipment"), can_get_to_floor:$("i_floor").value,
  };
  const {data} = await api("/api/intake", {patient_id:pid(), name:$("i_name").value, answers});
  $("i_msg").innerHTML = data.ok ? banner("ok", "Sent to your PT. They'll go through it with you at your first visit.") : banner("badb", errText(data));
  loadIntake();
};

// ---------- TAB 1: PT ----------
async function loadIntake(){
  const {data} = await api("/api/intake?patient_id=" + encodeURIComponent(pid()));
  $("pt_summary").textContent = data.intake ? data.intake.summary + `\n\n(received ${data.intake.created_at})` : "No intake yet for this patient.";
  const plan = (await api("/api/plan?patient_id=" + encodeURIComponent(pid()))).data.plan;
  $("pt_rows").innerHTML = "";
  (plan.length ? plan : [{exercise_id:"heel_slide",sets:3,reps:10},{exercise_id:"sit_to_stand",sets:3,reps:10},{exercise_id:"mini_squat",sets:3,reps:10}]).forEach(addRow);
}
function addRow(it = {exercise_id:"heel_slide", sets:3, reps:10}){
  const tr = document.createElement("tr");
  tr.innerHTML = `<td><select>${Object.entries(EX).map(([id,n]) => `<option value="${id}" ${id===it.exercise_id?"selected":""}>${esc(n)}</option>`).join("")}</select></td>
    <td class="n"><input type="number" min="1" max="5" value="${it.sets}"></td>
    <td class="n"><input type="number" min="1" max="30" value="${it.reps}"></td>
    <td class="n"><button class="ghost small" title="Remove">✕</button></td>`;
  tr.querySelector("button").onclick = () => tr.remove();
  $("pt_rows").appendChild(tr);
}
$("pt_add").onclick = () => addRow();
$("pt_refresh1").onclick = loadIntake;
$("pt_save").onclick = async () => {
  const items = [...$("pt_rows").querySelectorAll("tr")].map(tr => {
    const [sel, s, r] = tr.querySelectorAll("select, input");
    return {exercise_id:sel.value, sets:+s.value, reps:+r.value};
  });
  const {data} = await api("/api/plan", {patient_id:pid(), items, pt_name:$("pt_name").value});
  $("pt_msg").innerHTML = data.ok ? banner("ptb", `Plan saved: ${data.plan.map(p => `${esc(p.exercise_name)} ${p.sets} × ${p.reps}`).join(", ")}. The patient sees it in tab 2 and 3.`) : banner("badb", errText(data));
};

// ---------- TAB 2: daily exercise ----------
let session = null, selected = null;
$("seed").onclick = async () => {
  const {data} = await api("/api/test-patient", {patient_id:pid()});
  $("banner").innerHTML = banner("ok", `Test patient ready with ${data.plan.length} exercises from the PT.`);
  loadDb();
};
$("start").onclick = async () => {
  const {data} = await api("/api/checkin", {patient_id:pid(), pain_today:+$("pain").value, pain_location:$("loc").value});
  if (!data.ok) { $("banner").innerHTML = banner("badb", errText(data)); return; }
  session = data; selected = null;
  const cls = {as_planned:"ok", reduced:"warnb", rest:"badb"}[data.action];
  $("banner").innerHTML = banner(cls, `${esc(data.message)} <span style="font-weight:400">· check-in #${data.checkin_id}</span>`);
  $("plan").innerHTML = data.action === "rest" ? "" : data.exercises.map((e,i) => `
    <li data-i="${i}"><span><b>${esc(e.exercise_name)}</b> <small>${e.exercise_id}</small></span>
    <span class="dose">${e.target_reps !== e.planned_reps ? `<s>${e.sets} × ${e.planned_reps}</s>` : ""}${e.sets} × ${e.target_reps}</span></li>`).join("");
  $("planhint").textContent = data.action === "rest" ? "" : "Click an exercise to test it.";
  document.querySelectorAll("#plan li").forEach(li => li.onclick = () => pick(+li.dataset.i));
  if (data.action !== "rest") pick(0); else setButtons(false);
  loadDb();
};
function pick(i){
  selected = session.exercises[i];
  document.querySelectorAll("#plan li").forEach(li => li.classList.toggle("sel", +li.dataset.i === i));
  setButtons(true); simulate();
}
function setButtons(on){ ["sim","bad","send"].forEach(id => $(id).disabled = !on); }
function simulate(){
  const target = selected.sets * selected.target_reps;
  const done = Math.max(1, target - Math.floor(Math.random()*3));
  const base = 0.65 + Math.random()*0.25;
  const scores = Array.from({length:done}, (_,k) => +Math.min(1, Math.max(0, base - k*0.004 + (Math.random()-.5)*0.12)).toFixed(2));
  const errs = {}; const poor = scores.filter(s => s < 0.7).length;
  if (poor) errs[["trunk_lean","too_fast","short_range"][Math.floor(Math.random()*3)]] = poor;
  $("json").value = pretty({exercise_id:selected.exercise_id, sets_done:selected.sets, reps_done:done, form_scores:scores, error_counts:errs});
}
$("sim").onclick = simulate;
$("bad").onclick = () => {
  $("json").value = pretty({exercise_id:selected.exercise_id, sets_done:"three", reps_done:10, form_scores:[0.9, 1.4, 0.8], error_counts:{trunk_lean:-1}});
};
const tag = (kind, text) => `<span class="tag ${kind}">${text}</span>`;
$("send").onclick = async () => {
  let det;
  try { det = JSON.parse($("json").value); }
  catch(e){ $("res").textContent = "Your JSON doesn't parse: " + e.message; $("status").innerHTML = tag("badb","not sent"); return; }
  const body = {checkin_id: session.checkin_id, ...det, pain_after: +$("after").value};
  $("req").textContent = "POST /api/result\n\n" + pretty(body);
  const {status, data} = await api("/api/result", body);
  $("status").innerHTML = tag(data.ok ? "ok" : "badb", status + (data.ok ? " saved" : " rejected"));
  $("res").textContent = pretty(data);
  loadDb();
};
async function loadDb(){
  const {data} = await api("/api/db?patient_id=" + encodeURIComponent(pid()));
  $("table").innerHTML = data.logs.length ? `<table><thead><tr>
    <th>#</th><th>Date</th><th>Exercise</th><th class="n">Plan</th><th class="n">Today's target</th><th class="n">Done</th><th class="n">Form</th><th>Errors</th><th class="n">Pain after</th></tr></thead><tbody>
    ${data.logs.map(l => `<tr><td>${l.log_id}</td><td>${l.date}</td><td>${l.exercise_id}</td>
      <td class="n">${l.planned_sets} × ${l.planned_reps}</td><td class="n">${l.planned_sets} × ${l.target_reps}</td>
      <td class="n">${l.reps_done} reps</td><td class="n">${l.form_score ?? "—"}</td>
      <td>${Object.entries(JSON.parse(l.errors_json||"{}")).map(([k,v]) => `${esc(k)} ×${v}`).join(", ") || "—"}</td>
      <td class="n">${l.pain_after ?? "—"}</td></tr>`).join("")}</tbody></table>`
    : `<p class="empty">Nothing stored yet for this patient.</p>`;
  $("alerts").innerHTML = data.alerts.map(a => banner("badb", `Alert to PT · ${a.date} · ${esc(a.reason)}`)).join("");
}

// ---------- TAB 3: patient ----------
const KIND = {new:["ok","New"], paused:["warnb","Paused"], changed:["ptb","Changed"], same:["ghost","Same"]};
const DIR = {harder:["ok","Harder"], lighter:["warnb","Lighter"], mixed:["ptb","Mixed"], same:["ghost","Same"]};
const dirTag = d => d ? `<span class="dir ${DIR[d][0]}">${DIR[d][1]}</span>` : "";
const ACT = {progress:["ok","Step up"], add:["ok","New"], hold:["ghost","Same"], easier:["warnb","Lighter"], pause:["badb","Paused"]};
async function loadMyPlan(){
  const {data} = await api("/api/myplan?patient_id=" + encodeURIComponent(pid()));
  if (data.patient && GOALS.some(g => g.key === data.patient.goal)) $("my_goal").value = data.patient.goal;
  $("my_plan").innerHTML = data.plan.length ? data.plan.map(p => `<li><span><b>${esc(p.exercise_name)}</b></span><span>${p.sets} × ${p.reps}</span></li>`).join("")
    : `<p class="empty">Your PT hasn't set your plan yet.</p>`;
  const c = data.change;
  $("my_change").innerHTML = !c ? "" : banner("ptb", `Your PT updated your exercises${c.pt_note ? ": “" + esc(c.pt_note) + "”" : "."}`) +
    (c.explanation ? `<div class="expl">${dirTag(c.direction)}${esc(c.explanation)}</div>` : "") +
    `<ul class="plan">${c.changes.map(x => `<li><span><span class="tag ${(ACT[x.action] || KIND[x.kind])[0]}">${(ACT[x.action] || KIND[x.kind])[1]}</span> <b>${esc(x.exercise_name)}</b>
      <div class="reason">${x.kind === "same" ? "" : esc(x.reason)}</div></span>
      <span class="dose">${x.before && x.before !== x.after ? `<s>${x.before.replace(" x ", " × ")}</s>` : ""}${x.after ? x.after.replace(" x ", " × ") : "1 week off"}</span></li>`).join("")}</ul>`;
}
$("my_goal_save").onclick = async () => {
  const {data} = await api("/api/goal", {patient_id:pid(), goal:$("my_goal").value});
  $("my_msg").innerHTML = !data.ok ? banner("badb", errText(data))
    : (($("rv_msg").innerHTML = ""), banner("ok", data.drafted ? "Goal saved. A new plan for this goal was sent to your PT to check." : "Goal saved. Your PT will plan toward it after your first sessions."));
  loadReview();
};

// ---------- TAB 3: PT review ----------
let REQ = null;
async function loadReview(){
  const {data} = await api("/api/adjustment?patient_id=" + encodeURIComponent(pid()));
  REQ = data.pending;
  if (!REQ){ $("rv_body").innerHTML = `<p class="empty">No draft waiting for review.</p>`; return; }
  const s = REQ.summary, d = REQ.draft;
  const fmt = x => x ? x.replace(" x ", " × ") : "—";
  $("rv_body").innerHTML = `
    <h3>Week ${s.week.start} to ${s.week.end}${s.week_post_op != null ? ` · week ${s.week_post_op} after surgery` : ""}</h3>
    <div class="stats">
      <div class="stat"><b>${s.days_checked_in - s.days_rest}/7</b><span>days exercised</span></div>
      <div class="stat"><b>${s.pain_today_avg ?? "—"}</b><span>avg pain before</span></div>
      <div class="stat"><b>${s.days_reduced}</b><span>lighter days</span></div>
      <div class="stat"><b>${s.alerts.length}</b><span>alerts</span></div>
    </div>
    <label>Goal</label><b>${esc(d.goal_label || s.goal_label)}</b>
    <div class="expl">${dirTag(d.direction)}${esc(d.explanation)}</div>
    ${d.note_for_pt ? banner("warnb", "For you: " + esc(d.note_for_pt)) : ""}
    <h3>New plan</h3>
    <div id="rv_tbl"><table><thead><tr><th>Exercise</th><th class="n">Now</th><th class="n">New</th><th>Change</th></tr></thead>
    <tbody>${d.items.map((it,i) => `<tr data-i="${i}">
      <td><b>${esc(it.exercise_name)}</b><div class="reason">${esc(it.reason)}</div></td>
      <td class="n">${fmt(it.before)}</td>
      <td class="n"><span class="ro">${+it.sets === 0 ? "paused" : `${it.sets} × ${it.reps}`}</span>
        <span class="ro-edit prop"><input type="number" class="s" min="0" max="5" value="${it.sets}"> × <input type="number" class="r" min="0" max="15" value="${it.reps}"></span></td>
      <td><span class="tag ${ACT[it.action][0]}">${ACT[it.action][1]}</span></td></tr>`).join("")}</tbody></table></div>
    <p class="hint">Made by ${d.source === "llm" ? "the AI" : "rules (no AI yet)"}. The patient sees it only after you approve. <a href="#" id="rv_edit">Edit numbers first</a></p>
    <div class="grid2"><div><label for="rv_name">PT name</label><input type="text" id="rv_name" value="Dana Reyes, DPT"></div>
      <div><label for="rv_note">Note to the patient (optional)</label><input type="text" id="rv_note" placeholder="e.g. Great work on heel slides"></div></div>
    <div class="actions"><button class="ptbtn" id="rv_ok">Approve and send to patient</button><button class="ghost" id="rv_no">Reject</button></div>`;
  $("rv_edit").onclick = e => { e.preventDefault(); $("rv_tbl").classList.toggle("edit-on"); };
  $("rv_ok").onclick = () => decide(true);
  $("rv_no").onclick = () => decide(false);
}
async function decide(approve){
  const editing = $("rv_tbl").classList.contains("edit-on");
  const items = REQ.draft.items.map((it,i) => {
    if (!editing) return it;
    const tr = $("rv_tbl").querySelector(`tr[data-i="${i}"]`);
    return {...it, sets:+tr.querySelector(".s").value, reps:+tr.querySelector(".r").value};
  });
  const {data} = await api("/api/adjustment/decide", {request_id:REQ.request_id, approve, items, pt_name:$("rv_name").value, note:$("rv_note").value});
  $("rv_msg").innerHTML = data.ok ? banner("ptb", approve ? `Approved${data.pt_edited ? " with your edits" : ""}. The patient's plan is updated.` : "Rejected. The patient keeps the current plan.") : banner("badb", errText(data));
  loadReview(); loadMyPlan();
}
$("rv_demo").onclick = async () => {
  const {data} = await api("/api/demo-week", {patient_id:pid()});
  $("rv_msg").innerHTML = data.ok ? banner("ptb", "Loaded 7 days of sessions (week ending 2026-11-22). Now make the draft.") : banner("badb", errText(data));
  loadMyPlan();
};
$("rv_draft").onclick = async () => {
  const {data} = await api("/api/adjustment/draft", {patient_id:pid()});
  $("rv_msg").innerHTML = data.ok ? "" : banner("badb", errText(data));
  loadReview();
};
$("rv_refresh").onclick = loadReview;

// ---------- start ----------
(async () => {
  EX = (await api("/api/exercises")).data;
  GOALS = (await api("/api/goals")).data;
  document.querySelectorAll(".goalsel").forEach(sel => sel.innerHTML = GOALS.map(g => `<option value="${g.key}">${esc(g.label)}</option>`).join(""));
  $("i_goal").value = "home";
  loadIntake();
})();
</script>
</body>
</html>


In [ ]:
#@title 5. Start / restart the app
import sys, importlib, time, urllib.request
sys.path.insert(0, "/content/rehab_node")
try:                      # stop the old server if this cell was run before
    server.shutdown(); server.server_close()
except NameError:
    pass
import rehab_node, test_server
importlib.reload(rehab_node); importlib.reload(test_server)   # pick up code edits
server = test_server.start(host="0.0.0.0", port=8000)
for _ in range(20):       # wait until the server answers
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/exercises", timeout=1); break
    except Exception:
        time.sleep(0.5)
print("App running inside Colab. Database:", test_server.DB_PATH)

try:
    from google.colab.output import eval_js
    from IPython.display import display, HTML
    url = eval_js("google.colab.kernel.proxyPort(8000)")   # real https link to this Colab's port 8000
    display(HTML(f'<p style="font-size:16px">👉 <a href="{url}" target="_blank">Open the app in a new tab</a></p>'
                 f'<p>If the link shows an error, use the app below, or run step 6 for a public link.</p>'))
    from google.colab import output
    output.serve_kernel_port_as_iframe(8000, height=900)    # the app inside the notebook
except ImportError:
    print("Not on Colab: open http://127.0.0.1:8000")

## 6. *(Optional)* Public link
Use this when the **Figma/React frontend** or a teammate needs to call the API from outside Colab.
Put the printed URL in the frontend's `api.js` instead of `http://127.0.0.1:8000`.
⚠️ Anyone with the link can read and change the data, so use test data only. The link changes every time you run this cell.

In [ ]:
#@title 6. Make a public link (cloudflared)
import subprocess, re, time, os
CF = "/content/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["wget", "-q", "-O", CF,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod(CF, 0o755)
try:
    tunnel.terminate()   # stop the previous tunnel
except NameError:
    pass
log = open("/content/cloudflared.log", "w")
tunnel = subprocess.Popen([CF, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                          stdout=log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        PUBLIC_URL = m.group(0); break
print("Public URL:", PUBLIC_URL or "not ready, run this cell again")

## 7. Example: what the detection part sends
The pose/form model runs **one exercise**, then sends **one JSON** to `POST /api/result`. Video is never stored.
`form_scores` must have one value (0–1) per rep. Pain ≥ 7 after an exercise sends an alert to the PT.

In [ ]:
import requests, json
API = "http://127.0.0.1:8000"

# test patient with a demo plan (heel slide, sit-to-stand, mini squat: 3x10 each)
requests.post(API + "/api/test-patient", json={"patient_id": "u_test"})

# daily check-in: pain before the session -> today's plan with adjusted reps
today = requests.post(API + "/api/checkin", json={"patient_id": "u_test", "pain_today": 4, "pain_location": "front"}).json()
print(today["message"]); print(json.dumps(today["exercises"], indent=1))

# detection result for one exercise
result = {
    "checkin_id": today["checkin_id"],
    "exercise_id": "mini_squat",
    "sets_done": 3,
    "reps_done": 21,
    "form_scores": [0.9] * 15 + [0.7] * 6,     # one per rep
    "error_counts": {"trunk_lean": 2, "knee_valgus": 1},
    "pain_after": 3,
}
r = requests.post(API + "/api/result", json=result)
print(r.status_code, r.json())

## 8. Simulated patients (no real data yet)
`simulate.py` makes fake patients and plays out their weeks through the same functions the app uses
(check-in → detection JSON → weekly summary → rule draft → simulated PT approval).
Outputs: `plan_rows.csv` (one row per patient-week-exercise, with an empty `pt_label` column for the PT) and
`weekly_summaries.jsonl` (the exact input of the plan model). The numbers are made up: for testing and PT labelling only.

In [ ]:
%%writefile /content/rehab_node/simulate.py
"""Synthetic patients for testing the data flow and the plan-adjustment step.

There are no real patients yet, so this script makes fake ones. Each fake patient has hidden traits
(how fast pain drops, how good their form is, how often they skip) and a scenario such as "flare" or
"poor_form". The script then plays out their weeks through the SAME functions the app uses:

    daily_checkin -> log_exercise (one fake detection JSON per exercise) -> weekly_summary
    -> create_adjustment_request (rule draft) -> decide_request (a simulated PT approves the draft)

So the database fills up exactly as it would with real use, and every handoff between features gets tested.

Outputs (in --out):
  weekly_summaries.jsonl   one line per patient-week: the exact JSON the plan model receives
  plan_rows.csv            one row per patient-week-exercise: features + the rule draft's action
                           + an empty pt_label column for the PT to fill in (ground truth)
  The database (--db) holds all the raw rows (patients, plans, check-ins, exercise logs, alerts, drafts).

Run:  python simulate.py --patients 50 --weeks 6
The numbers are made up to look plausible. They are for testing and for PT labelling, not evidence.
"""
import argparse
import csv
import datetime as dt
import json
import os
import random

import rehab_node as rn

# How hard each exercise is on the knee (0-1). Harder exercises raise pain more when the dose is too high.
DIFFICULTY = {
    "heel_slide": 0.2, "straight_leg_raise": 0.3, "sitting_knee_bend": 0.2, "standing_knee_bend": 0.4,
    "seated_knee_extension": 0.3, "sit_to_stand": 0.5, "mini_squat": 0.6, "step_up": 0.8,
    "glute_bridge": 0.4, "side_lying_hip_abduction": 0.3,
}
# Form errors the detection model might report for each exercise.
ERRORS = {
    "heel_slide": ["partial_rom", "hip_lift"], "straight_leg_raise": ["knee_bent", "lift_too_high"],
    "sitting_knee_bend": ["partial_rom"], "standing_knee_bend": ["hip_flexion", "trunk_lean"],
    "seated_knee_extension": ["partial_rom", "too_fast"], "sit_to_stand": ["knee_valgus", "trunk_lean", "uses_hands"],
    "mini_squat": ["knee_valgus", "trunk_lean", "knees_past_toes"], "step_up": ["knee_valgus", "hip_drop"],
    "glute_bridge": ["hip_sag", "uneven_hips"], "side_lying_hip_abduction": ["hip_roll", "leg_forward"],
}
SCENARIOS = ["steady", "flare", "poor_form", "low_adherence", "early_goal", "plateau"]
START = dt.date(2026, 9, 7)  # first simulated Monday


def clip(x, lo, hi):
    return max(lo, min(hi, x))


def make_patient(rng, i):
    """Hidden traits. The app never sees these; they only drive the fake data."""
    scenario = SCENARIOS[i % len(SCENARIOS)]
    p = {
        "patient_id": f"sim_{i + 1:03d}",
        "scenario": scenario,
        "weeks_post_op_at_start": rng.randint(1, 6),
        "goal": rng.choice(["home", "walk", "stairs"]),
        "base_pain": rng.uniform(2.5, 5.0),     # pain before exercise in week 1
        "pain_drop": rng.uniform(0.80, 0.92),   # pain multiplied by this each week
        "capacity": rng.uniform(0.25, 0.45),    # how much load the knee tolerates in week 1
        "capacity_gain": rng.uniform(0.05, 0.10),
        "form": rng.uniform(0.72, 0.88),        # form score in week 1
        "form_gain": rng.uniform(0.01, 0.03),
        "adherence": rng.uniform(0.80, 0.97),   # chance of doing the session each day
        "flare_week": None, "flare_ex": None, "bad_form_ex": None,
    }
    if scenario == "flare":            # pain jumps for one or two weeks, caused by one exercise
        p["flare_week"] = rng.randint(2, 4)
    elif scenario == "poor_form":      # form on one exercise gets worse through a week
        p["form"] = rng.uniform(0.62, 0.72)
    elif scenario == "low_adherence":
        p["adherence"] = rng.uniform(0.35, 0.6)
    elif scenario == "early_goal":     # ambitious goal soon after surgery
        p["goal"], p["weeks_post_op_at_start"] = "jog", rng.randint(1, 3)
    elif scenario == "plateau":        # little improvement
        p["capacity_gain"], p["form_gain"], p["pain_drop"] = 0.0, 0.0, 0.99
    return p


def starter_plan(weeks_post_op):
    """What a PT might prescribe at the first visit."""
    if weeks_post_op <= 2:
        plan = [("heel_slide", 3, 10), ("straight_leg_raise", 2, 10), ("sit_to_stand", 2, 8)]
    else:
        plan = [("heel_slide", 3, 10), ("sit_to_stand", 3, 10), ("glute_bridge", 2, 10), ("mini_squat", 2, 8)]
    return [{"exercise_id": e, "exercise_name": rn.EXERCISES[e], "sets": s, "reps": r} for e, s, r in plan]


def fake_detection(rng, p, week, day, ex, pain_today):
    """One detection JSON for one exercise, like the pose/form model would send to POST /api/result."""
    eid = ex["exercise_id"]
    load = DIFFICULTY[eid] * ex["sets"] * ex["target_reps"] / 30          # ~0.1 (easy) to ~0.8 (hard)
    stress = load - (p["capacity"] + p["capacity_gain"] * week)            # > 0 means too much for the knee
    flare = p["flare_week"] is not None and week in (p["flare_week"], p["flare_week"] + 1) and eid == p["flare_ex"]

    n = ex["sets"] * ex["target_reps"]
    completion = clip(rng.gauss(0.97, 0.04) - max(0, stress) * 0.6 - (0.15 if flare else 0), 0.4, 1.0)
    reps_done = max(1, round(n * completion))

    form = p["form"] + p["form_gain"] * week - max(0, stress) * 0.4
    if eid == p["bad_form_ex"]:
        form -= 0.03 * day                                                  # gets worse through the week
    scores = [round(clip(form - 0.004 * r + rng.gauss(0, 0.04), 0, 1), 2) for r in range(reps_done)]

    avg = sum(scores) / len(scores)
    errors = {}
    if avg < 0.75:
        for name in rng.sample(ERRORS[eid], k=min(len(ERRORS[eid]), 1 + (avg < 0.6))):
            errors[name] = rng.randint(1, max(1, round((0.8 - avg) * 20)))

    pain_after = pain_today + max(0, stress) * 6 + rng.gauss(0.3, 0.7) + (rng.uniform(3, 5) if flare else 0)
    return {"exercise_id": eid, "sets_done": ex["sets"], "reps_done": reps_done, "form_scores": scores,
            "error_counts": errors, "pain_after": int(clip(round(pain_after), 0, 10))}


def simulate_patient(con, rng, p, weeks, use_llm=False):
    pid = p["patient_id"]
    for table in ("exercise_logs", "checkins", "alerts", "adjustment_requests", "plan_items", "patients"):
        con.execute(f"DELETE FROM {table} WHERE patient_id=?", (pid,))   # re-running gives a clean patient
    surgery = START - dt.timedelta(weeks=p["weeks_post_op_at_start"])
    rn.add_patient(con, pid, f"Sim {p['scenario']}", surgery.isoformat(), rng.choice(["left", "right"]), p["goal"])
    rn.set_plan(con, pid, starter_plan(p["weeks_post_op_at_start"]), approved_by="Sim PT", start_date=START.isoformat())

    out = []
    for week in range(weeks):
        plan_ids = [i["exercise_id"] for i in rn.current_plan(con, pid)]
        if p["scenario"] == "flare" and p["flare_ex"] is None:
            p["flare_ex"] = max(plan_ids, key=DIFFICULTY.get)    # the hardest exercise causes the flare
        if p["scenario"] == "poor_form":
            p["bad_form_ex"] = rng.choice(plan_ids) if week % 2 == 0 else None

        for day in range(7):
            date = START + dt.timedelta(weeks=week, days=day)
            if rng.random() > p["adherence"]:
                continue                                         # skipped day: no check-in
            flare_now = p["flare_week"] is not None and week in (p["flare_week"], p["flare_week"] + 1)
            pain = p["base_pain"] * p["pain_drop"] ** week + rng.gauss(0, 0.8) + (2.5 if flare_now else 0)
            pain = int(clip(round(pain), 0, 10))
            location = "none" if pain <= 3 else rng.choice(["front", "inner", "whole knee"])
            today = rn.daily_checkin(con, pid, pain, date.isoformat(), pain_location=location)
            if today["action"] == "rest":
                continue
            for ex in today["exercises"]:
                if ex["sets"] > 0:
                    rn.log_exercise(con, today["checkin_id"], fake_detection(rng, p, week, day, ex, pain))

        week_end = (START + dt.timedelta(weeks=week, days=6)).isoformat()
        req = rn.create_adjustment_request(con, pid, week_end, use_llm=use_llm)
        # Simulated PT: approves the draft as is, so next week's plan follows it.
        # Real PT decisions replace this once we have labels.
        # If the draft pauses everything, the simulated PT rejects it and keeps the old plan
        # (an empty plan would block the next check-in).
        keeps_some = any(i["sets"] > 0 for i in req["draft"]["items"])
        rn.decide_request(con, req["request_id"], approve=keeps_some, pt_name="Sim PT",
                          pt_note="" if keeps_some else "Draft paused every exercise; kept the current plan.")
        out.append({"scenario": p["scenario"], "week_index": week + 1, "summary": req["summary"], "draft": req["draft"]})
    return out


FIELDS = ["patient_id", "scenario", "week_index", "week_end", "week_post_op", "goal", "days_checked_in",
          "pain_today_avg", "days_reduced", "days_rest", "exercise_id", "planned", "days_done", "reps_done_share",
          "form_avg", "form_trend", "pain_after_avg", "pain_after_max", "high_pain_days", "error_total",
          "rule_action", "rule_new_dose", "draft_direction", "draft_added", "note_for_pt", "pt_label", "pt_new_dose", "pt_comment"]


def plan_rows(record):
    """Flatten one week into one row per exercise (the plan decision is made per exercise)."""
    s, d = record["summary"], record["draft"]
    actions = {i["exercise_id"]: i for i in d["items"]}
    rows = []
    for e in s["exercises"]:
        a = actions.get(e["exercise_id"], {})
        rows.append({
            "patient_id": s["patient_id"], "scenario": record["scenario"], "week_index": record["week_index"],
            "week_end": s["week"]["end"], "week_post_op": s["week_post_op"], "goal": s["goal"],
            "days_checked_in": s["days_checked_in"], "pain_today_avg": s["pain_today_avg"],
            "days_reduced": s["days_reduced"], "days_rest": s["days_rest"],
            "exercise_id": e["exercise_id"], "planned": e["planned"], "days_done": e["days_done"],
            "reps_done_share": e["reps_done_share"], "form_avg": e["form_avg"], "form_trend": e["form_trend"],
            "pain_after_avg": e["pain_after_avg"], "pain_after_max": e["pain_after_max"],
            "high_pain_days": e["high_pain_days"], "error_total": sum(e["error_counts"].values()),
            "rule_action": a.get("action", ""), "rule_new_dose": f"{a['sets']} x {a['reps']}" if a else "",
            "draft_direction": d["direction"],
            "draft_added": " ".join(i["exercise_id"] for i in d["items"] if i["action"] == "add"),
            "note_for_pt": d.get("note_for_pt", ""),
            "pt_label": "", "pt_new_dose": "", "pt_comment": "",
        })
    return rows


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--patients", type=int, default=50)
    ap.add_argument("--weeks", type=int, default=6)
    ap.add_argument("--seed", type=int, default=7, help="same seed = same data")
    ap.add_argument("--db", default="sim.db", help="kept separate from the app's rehab_test.db")
    ap.add_argument("--out", default="sim_out")
    ap.add_argument("--llm", action="store_true", help="use the LLM draft (needs OPENAI_API_KEY)")
    args = ap.parse_args()

    rng = random.Random(args.seed)
    con = rn.connect(args.db)
    os.makedirs(args.out, exist_ok=True)
    records = []
    for i in range(args.patients):
        records += simulate_patient(con, rng, make_patient(rng, i), args.weeks, use_llm=args.llm)

    with open(os.path.join(args.out, "weekly_summaries.jsonl"), "w") as f:
        for r in records:
            f.write(json.dumps({"scenario": r["scenario"], "week_index": r["week_index"], **r["summary"]}) + "\n")
    rows = [row for r in records for row in plan_rows(r)]
    with open(os.path.join(args.out, "plan_rows.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows(rows)

    count = lambda key, items: {k: sum(1 for x in items if x == k) for k in sorted(set(items))}
    print(f"{args.patients} patients x {args.weeks} weeks -> {len(records)} weekly summaries, {len(rows)} exercise rows")
    print("Draft direction per week:", count(None, [r["draft"]["direction"] for r in records]))
    print("Action per exercise:     ", count(None, [r["rule_action"] for r in rows]))
    print("Database:", os.path.abspath(args.db))
    print("Files:   ", os.path.abspath(args.out))


if __name__ == "__main__":
    main()


In [ ]:
#@title Run the simulator
PATIENTS = 50  #@param {type:"integer"}
WEEKS = 6  #@param {type:"integer"}
LOAD_INTO_APP = False  #@param {type:"boolean"}
#@markdown Tick `LOAD_INTO_APP` to put the fake patients in the app's database (look them up as sim_001, sim_002, ... in tab 3).
import subprocess, os, pandas as pd
db = os.environ["REHAB_DB"] if LOAD_INTO_APP else "/content/rehab_node/sim.db"
r = subprocess.run(["python", "simulate.py", "--patients", str(PATIENTS), "--weeks", str(WEEKS),
                    "--db", db, "--out", "/content/sim_out"], cwd="/content/rehab_node", capture_output=True, text=True)
print(r.stdout or r.stderr)
df = pd.read_csv("/content/sim_out/plan_rows.csv")
display(df.head(10))
try:
    from google.colab import files
    import shutil
    shutil.make_archive("/content/sim_out", "zip", "/content/sim_out")
    files.download("/content/sim_out.zip")
except ImportError:
    pass

### Endpoints
| Method | Path | Used by |
|---|---|---|
| POST | `/api/intake` | Tab 1: patient answers |
| POST | `/api/plan` | Tab 1: PT enters the plan |
| POST | `/api/checkin` | Tab 2: pain before session → today's reps |
| POST | `/api/result` | Detection: one JSON per exercise |
| POST | `/api/goal`, `/api/adjustment/draft`, `/api/adjustment/decide` | Tab 3: goal, weekly draft, PT approve/reject |
| GET | `/api/myplan`, `/api/adjustment`, `/api/db` | Plan view, PT review, database view |